# optimizer-loop-on-tensor — worked example 1: Gradient descent on a single (x, y) position tensor

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-loop-on-tensor`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

An optimizer can be applied to any tensor with `requires_grad=True`, not just model parameters. A common demonstration is gradient descent on a 2-D point `(x, y)` to minimize a scalar loss. The optimizer loop is identical to the model-training case: zero gradients, compute loss, call `.backward()`, call `.step()`.

## Worked solution

**Step 1 — Create the tensor to optimize.**
We start with `xy = t.tensor([3.0, -4.0], requires_grad=True)`. This is a single leaf tensor representing a 2-D coordinate.

**Step 2 — Pass the tensor to the optimizer.**
We write `opt = t.optim.SGD([xy], lr=0.1)`. The optimizer accepts a list with a single tensor. Passing `[xy]` (a list) rather than `(xy,)` is more conventional, but both work.

**Step 3 — Run the gradient descent loop.**
At each step: (a) zero gradients, (b) compute the loss (here the L2 distance from the origin, `(xy**2).sum()`), (c) call `.backward()` to populate `xy.grad`, (d) call `.step()` to update `xy.data`.

**Step 4 — Observe convergence.**
The loss should decrease each iteration as the point moves toward the origin. We print the coordinates and loss after each step.

In [ ]:
import torch as t

t.manual_seed(0)
xy = t.tensor([3.0, -4.0], requires_grad=True)
opt = t.optim.SGD([xy], lr=0.1)

for step in range(10):
    opt.zero_grad()
    loss = (xy ** 2).sum()   # L2 distance squared from origin
    loss.backward()
    opt.step()
    print(f'step {step+1:2d}: xy={xy.data.tolist()}, loss={loss.item():.4f}')

print(f'Final position: {xy.data.tolist()}')
assert (xy ** 2).sum().item() < (3.0**2 + 4.0**2), 'Point should have moved toward origin'